In [ ]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
from mppi import Parsers as P, Optics as O
from mppi.Utilities import Constants as Const
from mppi.Utilities import Utils
import matplotlib.pyplot as plt
import numpy as np
import os, re

# Transient absorption of LiF from the non-linear $\chi$

We compare the description of the transient absorption based on the pump and probe induced polarization (RT
simulations of D. Sangalli, one run for each pump-probe delay) with the one based on the equilibrium non-linear $\chi$
extracted from the `yambo_nl` simulations performed in `YamboNL_Analysis.ipynb`.

Reference: `../References/Pp_and_non_linear_functions.pdf` (C. Attaccalite, D. Sangalli).

The two descriptions are linked by the following equation
\begin{eqnarray}
\chi^{neq,(1)}[E_P](\omega) &=& \chi^{(1)}(\omega;\omega) + \nonumber \\
&+& \left[ \chi^{(2)}(\omega;\omega_P,\omega-\omega_P)\frac{E_{p}(\omega-\omega_P)}{E_p(\omega)} + 
\chi^{(2)}(\omega;-\omega_P,\omega+\omega_P)\frac{E_{p}(\omega+\omega_P)}{E_p(\omega)}
\right]E_{P}(\omega_P)  + \nonumber \\ &+&\left[   
\chi^{(3)}(\omega;\omega_P,\omega_P,\omega-2\omega_P)\frac{E_{p}(\omega-2\omega_P)}{E_p(\omega)} + 
\chi^{(3)}(\omega;\omega_P,-\omega_P,\omega) + \right. \nonumber \\ &&\left.
\chi^{(3)}(\omega;-\omega_P,-\omega_P,\omega+2\omega_P)\frac{E_{p}(\omega+2\omega_P)}{E_p(\omega)}
\right]E^2_{P}(\omega_P)
\end{eqnarray}

### Translation in terms of the MPPI quantities (MPPI >= 1.3)

`Xn_frequency_mixing(...).compute_Xn()[0][(n,m)]` at the probe frequency $\omega$ is the ratio between the component of
$P$ at $n\omega + m\omega_P$ and the product of the field amplitudes $E(\pm\omega)^{|n|} E(\pm\omega_P)^{|m|}$,
with $E(\omega) = (i E_0/2) e^{i\omega t_0}$ and $E(-\omega) = E(\omega)^*$ (Boyd, Nonlinear Optics, Sec. 1.3).
These quantities include the degeneracy factor: $\chi_{(1,\pm1)} = 2\chi^{(2)}$, $\chi_{(1,\pm2)} = 3\chi^{(3)}$
and the third order part of $\chi_{(1,0)}$ is $6\chi^{(3)}(\omega;\omega,\omega_P,-\omega_P)|E_P(\omega_P)|^2$.
The pump induced change of the response at the frequency $\omega$ reads

$$
\delta\chi(\omega) = \sum_{m=\pm1,\pm2} \chi_{(1,m)}(\omega - m\omega_P)\frac{E_p(\omega - m\omega_P)}{E_p(\omega)}
E_P(\mathrm{sign}(m)\,\omega_P)^{|m|} + \left[\chi_{(1,0)}(\omega) - \chi^{(1)}(\omega)\right]
$$

so the key $(1,m)$ is read at the probe frequency $\omega - m\omega_P$ (shifted grid) and the pump enters with its
complex amplitude $E_P(\omega_P) = (i E_{0P}/2) e^{i\omega_P t_{0P}}$. To be checked against `eval_dchi_neq`.

## Utility functions

In [ ]:
def eval_dchi_neq(omega,omega_P,tau,EP,x11,x1m1,x11m1,x12,x1m2):
    """
    We compute the variation of the neq absorption

    Args:
        omega : energy range of the probe (Hartree)
        omega_P : energy of the pump (Hartree)
        tau : time delay of the probe w.r.t. the pump (fs)
        EP : amplitude of the pump
        x11 : chi(omega+omega_P;omega_P,omega)
        x1m1 : chi(omega-omega_P;-omega_P,omega)
        x11m1 : chi(omega,omega_P,-omega_P,omega) extracted as the non-linear part
            of chi(omega)
        x12 : chi(omega+2*omega_P;omega_P,omega_P,omega)
        x1m2 : chi(omega-2*omega_P;-omega_P,-omega_P,omega)
    """
    domega = omega[1]-omega[0]
    ind_oP = round(omega_P/domega)
    omega_eff = omega[2*ind_oP:-2*ind_oP]
    phase = omega_P*tau*Const.FsToAu # corresponds to omega*t/hbar

    F2_1 = x11*np.exp(-1j*phase)
    F2_2 = x1m1*np.exp(1j*phase)
    F2_sum = F2_1[2*ind_oP:] + F2_2[:-2*ind_oP]
    F2_sum_cut = F2_sum[ind_oP:-ind_oP] # cut the part the is not present in the cubic terms

    F3_1 = x12*np.exp(-1j*2*phase)
    F3_2 = x11m1
    F3_3 = x1m2*np.exp(1j*2*phase)
    F3_sum = F3_1[4*ind_oP:] + F3_2[2*ind_oP:-2*ind_oP] + F3_3[:-4*ind_oP]

    dchi_neq = F2_sum_cut*EP+F3_sum*EP**2

    return omega_eff,dchi_neq


def build_delta_dict(path='.'):
    """
    Scan all folders inside `path` and build a dictionary
    using the value contained in 'deltaXX.XXfs' as key.

    Returns
    -------
    dict
        Dictionary of the form

            {
                delta_value: folder_name
            }

    Notes
    -----
    If multiple folders have the same delta value,
    the last one found will overwrite the previous one.
    """

    delta_dict = {}

    # Regular expression used to extract the delta value
    # Example match: delta10.00fs -> 10.00
    pattern = re.compile(r'delta([0-9]+\.?[0-9]*)fs')

    # Loop over all entries in the directory
    for folder in os.listdir(path):

        full_path = os.path.join(path, folder)

        # Process only directories
        if os.path.isdir(full_path):

            # Search for the delta pattern in the folder name
            match = pattern.search(folder)

            if match:

                # Convert extracted value to float
                delta = float(match.group(1))

                # Store folder name in the dictionary
                delta_dict[delta] = folder

    return delta_dict

## Extraction of the non-linear $\chi$ up to the third order

We compute the $\chi$ up to the third order (first order in the probe and second in the pump field) using the tools
of MPPI. We refer to the results

- `Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_10.0-20.0-nlensteps_201-bands_3-6-damp_0.1-sin`
- `pulse-E1_1e3-nlenrange_10.0-20.0-nlensteps_201-bands_3-6-damp_0.1-sin`
- `lresponse-bands_3-6-delta`

The first one is a P&p simulation with the intensity of the pump of 1e6 and the one of the probe of 1e3 kW/m$^2$, the
second one is a single pulse simulation with intensity of 1e3. The second one is used to separate the linear $\chi$ from
the cubic one $\chi^{(3)}(\omega;\omega,\omega_P,-\omega_P)$ in the same energy range. The delta run provides an
independent estimate of the linear $\chi$.

In [ ]:
run_dir = 'kx8_nb100_NoTr_E100'
file_Pp = os.path.join(run_dir,'Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_10.0-20.0-nlensteps_201-bands_3-6-damp_0.1-sin','ndb.Nonlinear')
file_pulse = os.path.join(run_dir,'pulse-E1_1e3-nlenrange_10.0-20.0-nlensteps_201-bands_3-6-damp_0.1-sin','ndb.Nonlinear')
file_delta = os.path.join(run_dir,'lresponse-bands_3-6-delta','ndb.Nonlinear')
data_Pp = P.YamboNLDBParser(file_Pp)
data_pulse = P.YamboNLDBParser(file_pulse)
data_delta = P.YamboNLDBParser(file_delta)

In [ ]:
freq_mix = O.Xn_frequency_mixing(data_Pp,X_order=(1,2),verbose=False)
single_freq = O.Xn_single_frequency(data_pulse,X_order=1,verbose=False)
chi_lr = single_freq.compute_Xn(set_units_of_measure=False)
chi_freqmix = freq_mix.compute_Xn(set_units_of_measure=False)

The energy range of the probe field is the same for both the simulations. We store the probe frequencies (Hartree),
the pump frequency (Hartree) and the pump amplitude (au)

In [ ]:
omega = freq_mix.probe_freqs
freqs = omega*Const.HaToeV
omega_P = freq_mix.pump_freq
EP = data_Pp.Efield2[0]['amplitude']
chi_lr[0].keys(),chi_freqmix[0].keys()

We compute the linear response within the freq mix and single pulse approach. Results are (almost) the same.

TODO: compare also with the linear response of the delta run (`Linear_Response`).

In [ ]:
Utils.Plot_ComplexArray(freqs,chi_lr[0][1],label='chi_lr(1)',data2=chi_freqmix[0][(1,0)],label2='chi_freqmix(1,0)')

Mixing terms linear in the pump. LiF is centrosymmetric, so these terms must vanish within the numerical accuracy.

In [ ]:
Utils.Plot_ComplexArray(freqs,chi_freqmix[0][(1,1)],label='chi_(1,1)',data2=chi_freqmix[0][(1,-1)],label2='chi_(1,-1)')

Mixing terms quadratic in the pump

In [ ]:
Utils.Plot_ComplexArray(freqs,chi_freqmix[0][(1,2)],label='chi_(1,2)',data2=chi_freqmix[0][(1,-2)],label2='chi_(1,-2)')

### The third order term at the probe frequency

There is one missing $\chi$, namely the $(\omega,\omega_P,-\omega_P)$ one, which is at the same output frequency of
the linear one. To extract it we remove the linear part from the polarization: since
$$
S(\omega,E_P) = \frac{P(\omega,E_P)}{E(\omega)} = \chi^{(1)}(\omega)+\chi_{(1,1,-1)}(\omega)|E_P(\omega_P)|^2
$$
the difference between the P&p and the probe-only results provides the third order term
$$
\chi_{(1,1,-1)}(\omega) = \frac{\chi_{(1,0)}(\omega)-\chi^{(1)}(\omega)}{|E_P(\omega_P)|^2} , \qquad
|E_P(\omega_P)|^2 = E_{0P}^2/4
$$
(with the MPPI conventions it includes the degeneracy factor 6).

In [ ]:
diffe = chi_freqmix[0][(1,0)] - chi_lr[0][1]
x11m1 = diffe/(EP**2/4)
Utils.Plot_ComplexArray(freqs,x11m1,label='chi_(1,1,-1)')

## Comparison of the NEQ vs NL-$\chi$ approaches

Now we compare the result for the transient absorption spectrum computed within the two approach.

To this scope we refer to the equation already discussed at the beginning of the notebook that (since we are interested in the variation w.r.t the equilibrium value)
can be written as

\begin{eqnarray}
\delta\chi^{neq,(1)}[E_P](\omega) &=& \chi^{neq,(1)}[E_P](\omega) - \chi^{(1)}(\omega;\omega) = \\
&& \left[ \chi^{(2)}(\omega;\omega_P,\omega-\omega_P)\frac{E_{p}(\omega-\omega_P)}{E_p(\omega)} + 
\chi^{(2)}(\omega;-\omega_P,\omega+\omega_P)\frac{E_{p}(\omega+\omega_P)}{E_p(\omega)}
\right]E_{P}(\omega_P)  + \nonumber \\ &+&\left[   
\chi^{(3)}(\omega;\omega_P,\omega_P,\omega-2\omega_P)\frac{E_{p}(\omega-2\omega_P)}{E_p(\omega)} + 
\chi^{(3)}(\omega;\omega_P,-\omega_P,\omega) + \right. \nonumber \\ &&\left.
\chi^{(3)}(\omega;-\omega_P,-\omega_P,\omega+2\omega_P)\frac{E_{p}(\omega+2\omega_P)}{E_p(\omega)}
\right]E^2_{P}(\omega_P)
\end{eqnarray}

In order to write an equation useful to compute the $\delta\chi^{neq}$ we analyze the previous formula addend by addend.

We parametrize the $\omega$ dependence in terms of the values of the probe, so that the two addends at the second order reads

$$
 \chi^{(2)}(\omega+\omega_P;\omega_P,\omega)\frac{E_{p}(\omega)}{E_p(\omega+\omega_P)} + 
\chi^{(2)}(\omega-\omega_P;-\omega_P,\omega)\frac{E_{p}(\omega)}{E_p(\omega-\omega_P)}
$$

Assuming tha the probe is a delta centred at $t_0$ in the time domain, its FT is given by $e^{i\omega t_0}$ so the ratios of probe fields simplify to

$$
\chi^{(2)}(\omega+\omega_P;\omega_P,\omega)e^{-i\omega_P \, t_0} + 
\chi^{(2)}(\omega-\omega_P;-\omega_P,\omega)e^{i\omega_P \, t_0}
$$

In the same way we can rewrite the third order addends as

$$
\chi^{(3)}(\omega;\omega_P,-\omega_P,\omega) +
\chi^{(3)}(\omega+2\omega_P;\omega_P,\omega_P,\omega)e^{-2i\omega_P \, t_0} + 
\chi^{(3)}(\omega-2\omega_P;-\omega_P,-\omega_P,\omega)e^{2i\omega_P \, t_0}
$$

The function `eval_dchi_neq` (see the utility functions) builds these terms on the common frequency range.
It has still to be checked against the formula in terms of the MPPI quantities written above.

In [ ]:
x11 = chi_freqmix[0][(1,1)]
x1m1 = chi_freqmix[0][(1,-1)]
x12 = chi_freqmix[0][(1,2)]
x1m2 = chi_freqmix[0][(1,-2)]

RT results of D. Sangalli (kx16): absorption spectrum for each pump-probe delay

In [ ]:
path = 'Davide_data/20250312_Report_simulations_data_kx16_ball/'
deltas_folder = build_delta_dict(path=path)
deltas = sorted(deltas_folder.keys())
deltas

In [ ]:
tau = deltas[0]

data = P.YamboOutputParser.from_file(os.path.join(path,deltas_folder[tau],'o-abs_0-50eV_sm0.6eV.YPP-eps_along_E'))['YPP-eps_along_E']
energy = data['col1']
eps_im = data['col2']
omega_eff, dchi_neq = eval_dchi_neq(omega,omega_P,tau,EP,x11,x1m1,x11m1,x12,x1m2)

abs_chi = np.imag(dchi_neq)/max(np.imag(dchi_neq))

plt.plot(energy,eps_im/max(eps_im),label='RT, delay %s fs'%tau)
plt.plot(omega_eff*Const.HaToeV,abs_chi,label='NL chi')
plt.xlim(10,20)
plt.legend()